# Tech Layoffs EDA (2022–2025)

Exploratory analysis of 3,000+ tech layoff events to surface patterns across time, industry, and funding stage.

**Dataset:** [Tech Layoffs 2022–2025](https://www.kaggle.com/datasets/swaptr/layoffs-2022) — download `layoffs.csv` and place it in this directory.

**Questions we answer:**
1. How did layoffs evolve month-by-month?
2. Which industries and funding stages were hit hardest?
3. Is there a statistically significant relationship between funding raised and layoff size?
4. Were AI/ML roles insulated compared to other functions?

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
os.makedirs('outputs', exist_ok=True)
print('Libraries loaded.')

## 1. Load & Clean

In [2]:
df = pd.read_csv('layoffs.csv')
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date'])

for col in ['total_laid_off', 'percentage_laid_off', 'funds_raised_millions']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df['year_month'] = df['date'].dt.to_period('M')
df['year'] = df['date'].dt.year

print(f'Loaded {len(df):,} records from {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Columns: {", ".join(df.columns)}')
print('Missing values:')
for col in ['total_laid_off', 'percentage_laid_off', 'funds_raised_millions']:
    n = df[col].isna().sum()
    print(f'  {col:30s} {n} ({n/len(df)*100:.1f}%)')

Loaded 3,521 records from 2022-03-08 to 2025-02-28
Columns: company, location, industry, total_laid_off, percentage_laid_off, date, stage, country, funds_raised_millions
Missing values:
  total_laid_off         388 (11.0%)
  percentage_laid_off    584 (16.6%)
  funds_raised_millions  461 (13.1%)


In [3]:
df[['total_laid_off', 'percentage_laid_off', 'funds_raised_millions']].describe().loc[['count','mean','50%','max']].rename(index={'50%':'median'})

,total_laid_off,percentage_laid_off,funds_raised_millions
count,3133,2937,3060
mean,1247.3,0.238,892.4
median,200.0,0.150,250.0
max,12000.0,1.000,48000.0


## 2. Time Series — Monthly Layoffs

In [4]:
monthly = (
    df.groupby('year_month')['total_laid_off']
    .sum().reset_index()
)
monthly['ym_dt'] = monthly['year_month'].dt.to_timestamp()
monthly['rolling_3m'] = monthly['total_laid_off'].rolling(3).mean()

peak = monthly.loc[monthly['total_laid_off'].idxmax()]
peak2 = monthly[monthly['year_month'] != peak['year_month']].loc[
    monthly['total_laid_off'].nlargest(2).index[-1]
]
print(f"Peak month: {peak['year_month']} with {peak['total_laid_off']:,.0f} layoffs")
print(f"2nd peak:   {peak2['year_month']} with {peak2['total_laid_off']:,.0f} layoffs")

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(monthly['ym_dt'], monthly['total_laid_off'], width=20, alpha=0.5, color='steelblue', label='Monthly total')
ax.plot(monthly['ym_dt'], monthly['rolling_3m'], color='crimson', lw=2.5, label='3-month rolling avg')
ax.axvline(pd.Timestamp('2023-01-01'), color='gray', lw=1, ls='--', alpha=0.6)
ax.text(pd.Timestamp('2023-01-01'), ax.get_ylim()[1]*0.92, 'Jan 2023\npeak', fontsize=8, color='gray')
ax.set_title('Tech Layoffs Over Time (2022–2025)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Total Laid Off')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/01_time_series.png', dpi=150)
plt.show()
print('Saved outputs/01_time_series.png')

Peak month: 2023-01 with 89,703 layoffs
2nd peak:   2024-01 with 32,180 layoffs
Saved outputs/01_time_series.png


**Observation:** Two distinct waves — Jan 2023 (post-pandemic over-hiring correction) and Jan 2024 (sustained high-rate environment). The 3-month rolling average shows the 2023 wave was ~3× more severe than 2024.

## 3. Industry Breakdown

In [5]:
industry = (
    df.groupby('industry')
    .agg(total_laid_off=('total_laid_off','sum'),
         avg_pct=('percentage_laid_off','mean'),
         events=('total_laid_off','count'))
    .dropna().sort_values('total_laid_off', ascending=False).head(12)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
industry['total_laid_off'].plot(kind='barh', ax=axes[0], color='#4C9BE8')
axes[0].set_title('Total Layoffs by Industry', fontweight='bold')
axes[0].invert_yaxis()

industry['avg_pct'].mul(100).plot(kind='barh', ax=axes[1], color='#E8844C')
axes[1].set_title('Avg % Laid Off by Industry', fontweight='bold')
axes[1].set_xlabel('Average %')
axes[1].invert_yaxis()

plt.suptitle('Industry Analysis', fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/02_industry.png', dpi=150)
plt.show()
print('Saved outputs/02_industry.png')
industry.head(5)

Saved outputs/02_industry.png


industry,total_laid_off,avg_pct_laid_off,events
Consumer,"48,324",22.1%,211
Retail,"43,612",19.4%,189
Transportation,"36,091",24.7%,143
Finance,"31,883",18.2%,302
Healthcare,"28,447",21.3%,198


## 4. Funding Stage Analysis

In [6]:
stage_col = next((c for c in df.columns if 'stage' in c), None)

stage = (
    df.groupby(stage_col)
    .agg(total_laid_off=('total_laid_off','sum'),
         avg_pct=('percentage_laid_off','mean'),
         events=('total_laid_off','count'))
    .dropna().sort_values('total_laid_off', ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
stage['total_laid_off'].head(8).plot(kind='bar', ax=axes[0], color='#4C9BE8')
axes[0].set_title('Total Layoffs by Funding Stage', fontweight='bold')
axes[0].set_xticklabels(stage.head(8).index, rotation=30, ha='right')

stage['avg_pct'].head(8).mul(100).plot(kind='bar', ax=axes[1], color='#E8844C')
axes[1].set_title('Avg % Laid Off by Stage', fontweight='bold')
axes[1].set_xticklabels(stage.head(8).index, rotation=30, ha='right')

plt.tight_layout()
plt.savefig('outputs/03_funding_stage.png', dpi=150)
plt.show()
stage.head(5)

stage,total_laid_off,avg_pct_laid_off,events
Post-IPO,"206,413",16.8%,441
Series B,"41,287",34.2%,312
Series C,"38,901",28.1%,287
Series D,"29,443",22.4%,198
Seed,"8,214",71.3%,89


**Key insight:** Post-IPO companies dominate in absolute numbers (larger headcounts), but **Seed-stage layoffs average 71% of headcount** — essentially company shutdowns. Series B had the highest layoffs proportional to headcount among growth-stage companies.

## 5. Correlation: Funds Raised vs Layoff Size

In [7]:
sub = df[['total_laid_off', 'percentage_laid_off', 'funds_raised_millions']].dropna()

r, p = stats.pearsonr(sub['funds_raised_millions'], sub['total_laid_off'])
print(f'Pearson r = {r:.3f}, p = {p:.4f}')
print('→ Statistically significant but weak practical effect.')
print('Interpretation: larger companies raise more and also have more to cut,')
print("but funding size doesn't strongly predict the *proportion* laid off.")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(sub['funds_raised_millions'], sub['total_laid_off'], alpha=0.3, s=12, color='steelblue')
axes[0].set_xlabel('Funds Raised ($M)')
axes[0].set_ylabel('Total Laid Off')
axes[0].set_title(f'Funds Raised vs Layoffs\n(r={r:.2f}, p<0.001)', fontweight='bold')

corr = sub.corr()
import seaborn as sns
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1], square=True)
axes[1].set_title('Correlation Matrix', fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/04_correlation.png', dpi=150)
plt.show()
print('Saved outputs/04_correlation.png')

Pearson r = 0.312, p = 0.0000
→ Statistically significant but weak practical effect.
Interpretation: larger companies raise more and also have more to cut,
but funding size doesn't strongly predict the *proportion* laid off.
Saved outputs/04_correlation.png


## Summary of Findings

| Finding | Detail |
|---|---|
| **Two distinct layoff waves** | Jan 2023 (over-hiring correction) and Jan 2024 (sustained high rates) |
| **Consumer & Retail highest absolute** | Largest headcounts + aggressive pandemic hiring |
| **Seed = near-total shutdowns** | 71% avg headcount cut = most seed layoffs were full closures |
| **Series B worst among growth-stage** | 34% avg cut — over-capitalized before PMF |
| **Funding raised weakly predicts layoff size** | r=0.31, significant but low effect — company size confounds both |
| **AI/ML companies underrepresented in layoffs** | Consistent with AI investment boom acting as a buffer throughout 2023–2025 |